
# Проверка гипотезы для сервиса Яндекс Книги


# Цели и задачи 


**Цель проекта** — Проверить гипотезу о том, что пользователи из Санкт-Петербурга в среднем проводят больше времени за чтением и прослушиванием контента в приложении Яндекс Книги, чем пользователи из Москвы.

**Задачи проекта**
- Загрузить и подготовить данные.
- Провести описательный анализ, сравнить размеры групп и их статистики (средние значения, медианы, разброс, распределения метрик) для пользователей Москвы и Санкт‑Петербурга, а также для групп A и B A/B‑теста.
- Выбрать и применить корректные статистические тесты: t‑тест для сравнения средних часов активности между городами и тест для сравнения долей (конверсий) между группами интерфейсного A/B‑теста, задать уровень значимости и посчитать p‑value.  
- Интерпретировать результаты: сделать выводы о принятии или отклонении гипотез, сформулировать возможные причины наблюдаемых эффектов и оценить их практическую значимость для бизнеса.



# Описание данных 

Данные представлены несколькими датасетами:

- таблица с данными пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания)

- таблица участников тестов.

- таблица, в которой собраны события 2020 года;


##  Проверка гипотезы в Python

Данные предобработаны в SQL, и теперь  готовы для проверки гипотезы в Python. Загружаем данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv.

Проверим наличие дубликатов в идентификаторах пользователей. Сравним размеры групп, их статистики и распределение.

Гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.



## Загрузка данных и знакомство с ними

Загружаем данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания).

In [1]:
import pandas as pd 

import numpy as np

from scipy.stats import ttest_ind

from statsmodels.stats.proportion import proportions_ztest

from statsmodels.stats.power import NormalIndPower


In [2]:
df_knigi = pd.read_csv('/datasets/yandex_knigi_data.csv', index_col=0)

print(df_knigi.info())
display(df_knigi.head(5))

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8784 entries, 0 to 8783
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   city    8784 non-null   object 
 1   puid    8784 non-null   int64  
 2   hours   8784 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 274.5+ KB
None


,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434


In [3]:
print(df_knigi['city'].value_counts())

Москва             6234
Санкт-Петербург    2550
Name: city, dtype: int64


In [4]:
# Дубликаты
dupl = df_knigi.duplicated(subset='puid').sum()
print(dupl)

244


In [5]:
# Выводим дубликаты
show_dupl = df_knigi[df_knigi.duplicated(subset='puid', keep = False)]
show_dupl = show_dupl.sort_values(by = 'puid')

display(show_dupl.head(10))

,city,puid,hours
35,Москва,2637041,10.317371
6247,Санкт-Петербург,2637041,3.883926
134,Москва,9979490,32.415573
6274,Санкт-Петербург,9979490,1.302997
145,Москва,10597984,42.931506
6279,Санкт-Петербург,10597984,9.041320
150,Москва,10815097,9.086655
6283,Санкт-Петербург,10815097,0.323291
187,Москва,13626259,21.104167
6300,Санкт-Петербург,13626259,1.648434


In [6]:
# Агрегация часов по пользователю и городу
knigi_agg = (df_knigi
               .groupby(['puid','city'], as_index= False)
               .agg({'hours': 'sum'}))

print(knigi_agg.head(10))

     puid             city       hours
0    9668           Москва   26.167776
1   16598           Москва   82.111217
2   80401           Москва    4.656906
3  104923  Санкт-Петербург   60.353889
4  140205           Москва    1.840556
5  146427  Санкт-Петербург    0.469559
6  248755           Москва  151.326434
7  295646  Санкт-Петербург    1.258954
8  352567           Москва    8.206369
9  439493           Москва    0.857758


In [7]:
# Для пользователя оставляем город с максимальной активностью
knigi_clean = (
    knigi_agg
    .sort_values(by=['puid', 'hours'], ascending=[True, False])
    .drop_duplicates(subset='puid', keep='first')
)

# Проверяем дубликаты повторно
dupl = knigi_clean.duplicated(subset='puid').sum()

print(dupl)

0


In [8]:
display(knigi_clean['city'].value_counts())
display(knigi_clean.groupby('city')['hours'].describe())

Москва             6099
Санкт-Петербург    2441
Name: city, dtype: int64

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Москва,6099.0,11.075176,37.229705,0.000022,0.058889,0.930357,6.025852,857.209373
Санкт-Петербург,2441.0,12.028376,40.521834,0.000025,0.075278,1.025801,6.880545,978.764775


In [9]:
#Разделение выборок

spb = knigi_clean[knigi_clean['city'] == 'Санкт-Петербург']['hours']
moscow = knigi_clean[knigi_clean['city'] == 'Москва']['hours']

print(spb, moscow)

3       60.353889
5        0.469559
7        1.258954
11       0.089076
12       0.334019
          ...    
8753     5.879444
8759    45.069222
8766     0.211944
8774     4.311841
8782    20.847222
Name: hours, Length: 2441, dtype: float64 0        26.167776
1        82.111217
2         4.656906
4         1.840556
6       151.326434
           ...    
8778      0.003319
8779      0.081234
8780      0.024736
8781      0.209018
8783      0.609346
Name: hours, Length: 6099, dtype: float64


В ходе предобработки данных, были выявлены пользователи, относящиеся к нескольким городам, создающие дубликаты. Для устранения дублирования, каждому пользователю был сопоставлен город с наибольшей суммарной активностью, после чего данные были агрегированы до одного наблюдения на пользователя.

## Проверка гипотезы в Python

Продублируем гипотезу: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. 

Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [10]:
stat, p_value = ttest_ind(
    spb,
    moscow,
    alternative='greater',
    equal_var=False
)

print(stat, p_value)


1.004793916530622 0.15752715242320128


In [11]:
#Принятие решения

alpha = 0.05

if p_value < alpha:
    print('Отвергаем H₀: средняя активность пользователей СПб выше, чем в Москве')
else:
    print('Нет оснований отвергать H₀: статистически значимых различий не выявлено')


Нет оснований отвергать H₀: статистически значимых различий не выявлено


Так как полученное значение p-value выше уровня статистической значимости 0,05, нулевая гипотеза не была отвергнута.
Это означает, что среднее время активности пользователей в Санкт-Петербурге выше, чем в Москве, но не значительно.

In [12]:
# Проверка перед тестом
print('СПБ', spb.shape[0])
print('Москва', moscow.shape[0])

СПБ 2441
Москва 6099


## Аналитическая записка


В ходе анализа проверялись следующие гипотезы:

•	H₀: средняя активность пользователей в Москве и Санкт-Петербурге не различается

•	H₁: средняя активность пользователей в Санкт-Петербурге выше, чем в Москве

Для проверки гипотез был использован односторонний t-test для двух независимых выборок (Welch t-test).
Выбор данного теста обусловлен несколькими факторами: сравниваются средние значения в двух независимых группах, размеры выборок различаются, дисперсии в группах могут быть неодинаковыми.

Установлен следующий уровень статистической значимости: α = 0,05.


В результате проведения t-теста, было получено значение:

•	t-статистика = 1.00

•	p-value = 0.16

**Вывод**

Так как полученное значение p-value (0.16) превышает уровень статистической значимости (0.05), нет оснований отвергать нулевую гипотезу.
Это означает, что статистически значимых доказательств того, что пользователи из Санкт-Петербурга проводят больше времени за чтением и прослушиванием контента, чем пользователи из Москвы, не выявлено.

Возможные причины полученного результата

1.	Фактическая разница в средней активности между пользователями Москвы и Санкт-Петербурга может быть небольшой и недостаточной для выявления статистически значимого эффекта при имеющемся объёме данных.
2.	Поведение пользователей в крупных городах может быть схожим из-за одинакового уровня доступности сервиса, образа жизни и структуры аудитории, что приводит к близким значениям средней активности.

